## 05_summary: Tables and Figures

Builds the manuscript's tables and figures from the outputs of 01–04c (`cohort_master`,
`wearable_features` and the `results/` files). Any count, percentage or estimate covering
fewer than 20 participants is shown as `<20`. Local copies go to `res/nb05/`; the results
tables are also saved to `results/` in the output bucket.

`HC` in code and file names is the non-clinical control group (NCC) of the manuscript.

## Task 0: Setup

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.utils import *
from src.analysis import (
    ETM_TASKS, TASK_CONFIG, PHENOTYPES, MIN_CELL,
    prepare_cohort, qc_pass_mask, fit_normative_model, normative_sample,
    fitbit_sample, hc_fitbit_sample, phenotype_complete, residualise_phenotype,
)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'axes.titlesize': 11, 'axes.labelsize': 9,
                     'xtick.labelsize': 8, 'ytick.labelsize': 8})

OVERWRITE = True
RES_OUT   = 'nb05'
DPI = 300

PHENO_LABELS = {
    'mean_steps':          'Steps (mean/day)',
    'mean_waso':           'WASO (min)',
    'STV':                 'STV (h)',
    'mean_sleep_duration': 'TST (min)',
}

SEASONS = {
    'Spring': [3, 4, 5],
    'Summer': [6, 7, 8],
    'Autumn': [9, 10, 11],
    'Winter': [12, 1, 2],
}

MED_CLASSES = [
    'ssri', 'snri', 'lithium', 'antipsychotic',
    'anticonvulsant', 'maoi', 'tca', 'atypical_antidepressant',
]
MED_LABELS = {
    'ssri':                    'SSRI',
    'snri':                    'SNRI',
    'lithium':                 'Lithium',
    'antipsychotic':           'Antipsychotic',
    'anticonvulsant':          'Anticonvulsant',
    'maoi':                    'MAOI',
    'tca':                     'TCA',
    'atypical_antidepressant': 'Atypical antidepressant',
}
plt.rcParams.update({
    'font.size': 14,          # Global base text size fallback
    'axes.titlesize': 16,     # Individual subplot titles
    'axes.labelsize': 13,     # X and Y axis labels
    'xtick.labelsize': 15,     # X-axis tick values (numbers)
    'ytick.labelsize': 15,     # Y-axis tick values (numbers)
    'legend.fontsize': 16,     # Legend text size
    'figure.titlesize': 18    # Global figure suptitle (if used)
})

GROUP_COLORS = {'MDD': '#1f77b4', 'BD': '#d62728', 'HC': '#2ca02c'}

In [ ]:
def desc_stats(series):
    return f'{series.mean():.1f} ({series.std():.1f})'


SUPPRESSED = '<20 (suppressed)'


def pct_female(df):
    """% female among Female/Male; suppressed when the female count is < 20."""
    n_f = int((df['sex'] == 'Female').sum())
    n_m = int((df['sex'] == 'Male').sum())
    if n_f < 20:
        return SUPPRESSED
    return f'{n_f:,} ({100*n_f/(n_f + n_m):.1f}%)'


def pct_of_total(df, col, value):
    """Count of df[col] == value as a percentage of len(df); suppressed if the count < 20."""
    n = int(df[col].eq(value).fillna(False).sum())
    if n < 20:
        return SUPPRESSED
    return f'{n:,} ({100*n/len(df):.1f}%)'


def pct_touch(df, task):
    """% on a touchscreen device; suppressed when the touchscreen count is < 20."""
    col = f'touch_{task}'
    if col not in df.columns:
        return 'N/A'
    return pct_of_total(df, col, True)


def category_rows(df, col, prefix, categories):
    """One {prefix: category} -> pct_of_total row per category. Falls back to a single
    {prefix: 'N/A'} row when categories is empty (e.g. no non-missing values in this
    sample)."""
    if not categories:
        return {prefix: 'N/A'}
    return {f'{prefix}: {cat}': pct_of_total(df, col, cat) for cat in categories}


def season_rows(df, task):
    """Season of the test date, as % of the sample; each season is suppressed when its
    own count is < 20."""
    months = pd.to_datetime(df[f'etm_date_{task}']).dt.month
    counts = {s: int(months.isin(ms).sum()) for s, ms in SEASONS.items()}
    return {f'Season: {s}': f'{100*n/len(df):.1f}%' if n >= 20 else SUPPRESSED
            for s, n in counts.items()}


def bmi_desc(df, task):
    """Mean (SD) BMI with the number of participants missing BMI; that count is
    suppressed when it is 1-19."""
    col = f'bmi_{task}'
    if col not in df.columns:
        return 'N/A'
    n_miss = int(df[col].isna().sum())
    miss_s = f'{n_miss}' if n_miss == 0 or n_miss >= 20 else '<20'
    return desc_stats(df[col].dropna()) + f' [N missing={miss_s}]'


def med_pct(df, cls, task):
    col = f'on_{cls}_{task}'
    if col not in df.columns:
        return 'N/A'
    n_on  = int((df[col] == 1.0).sum())
    n_tot = len(df)
    if n_on < 20:
        return SUPPRESSED
    return f'{n_on:,} ({100*n_on/n_tot:.1f}%)'

In [ ]:
# Load inputs
cohort        = load_parquet('cohort/cohort_master.parquet')
wf            = load_parquet('features/wearable_features.parquet')
model_results = load_parquet('results/model_results.parquet')
hc_model_results = load_parquet('results/model_results_hc_sensitivity.parquet')
attrition     = load_parquet('cohort/attrition.parquet')

prepare_cohort(cohort)

# Fit all HC models and build all samples upfront
models     = {}
ab_mdd     = {}
ab_bd      = {}
c_mdd      = {}
c_bd       = {}
c_hc       = {}

for task in ETM_TASKS:
    m, sigma, hc   = fit_normative_model(cohort, task)
    ab_m           = normative_sample(cohort, task, 'MDD', m, sigma)
    ab_b           = normative_sample(cohort, task, 'BD',  m, sigma)
    cm             = fitbit_sample(ab_m, task, wf, 'MDD')
    cb             = fitbit_sample(ab_b, task, wf, 'BD')
    ch             = hc_fitbit_sample(hc, task, wf)
    models[task]   = (m, sigma, hc)
    ab_mdd[task]   = ab_m
    ab_bd[task]    = ab_b
    c_mdd[task]    = cm
    c_bd[task]     = cb
    c_hc[task]     = ch
    print(f'{task}:  HC={len(hc):,}  MDD_AB={len(ab_m):,}  BD_AB={len(ab_b):,}  '
          f'MDD_C={len(cm):,}  BD_C={len(cb):,}  HC_C={len(ch):,}')

# wf_avg (Task 5): per phenotype, each participant modelled for it (MDD and HC wearable
# samples) with their value averaged over the tasks they enter; NaN outside that sample.
_pheno_avgs = []
for pheno in PHENOTYPES:
    _rows = pd.concat(
        [phenotype_complete(s[task], pheno)[['person_id', 'group', pheno]]
         for s in (c_mdd, c_hc) for task in ETM_TASKS],
        ignore_index=True,
    )
    _pheno_avgs.append(_rows.groupby(['person_id', 'group'])[pheno].mean())
wf_avg = pd.concat(_pheno_avgs, axis=1).reset_index()
for grp in ['MDD', 'HC']:
    _g = wf_avg[wf_avg['group'] == grp]
    print(f'wf_avg {grp}: {len(_g):,} unique participants in a Fitbit analytic sample '
          + ', '.join(f'{p} {int(_g[p].notna().sum()):,}' for p in PHENOTYPES))

## Task 1: Attrition (eTable 3)

Retention cascades for MDD, BD and NCC, per task.

In [ ]:
def _safe_n(n):
    return n if n >= 20 else '<20 (suppressed)'


def build_mdd_cascade(task, cohort, wf):
    rows = []
    base = cohort[cohort['group'] == 'MDD'].copy()

    step0_label = f'3a. MDD qualifying — {task}'
    attr_row = attrition[
        (attrition['step'] == step0_label) & (attrition['group'] == 'MDD')
    ]
    n0 = int(attr_row['n_after'].values[0]) if len(attr_row) > 0 else \
         int((base[f'mdd_eligible_{task}'].fillna(0) == 1).sum())
    rows.append(('MDD-eligible', n0))

    cur = base[base[f'mdd_eligible_{task}'].fillna(0) == 1].copy()

    cur = cur[cur[f'etm_washout_pass_{task}'].fillna(False).astype(bool)]
    rows.append(('EtM washout pass', len(cur)))

    qc  = qc_pass_mask(cur, task)
    cur = cur[qc]
    rows.append(('QC pass', len(cur)))

    cur = cur[cur[f'covs_eligible_normative_{task}']]
    rows.append(('covs_eligible_normative_{task} → Analysis Normative', len(cur)))

    cur = cur[cur[f'fitbit_sufficient_{task}'].fillna(False).astype(bool)]
    rows.append(('Fitbit sufficient', len(cur)))

    cur = cur[cur[f'wearable_washout_pass_{task}'].fillna(False).astype(bool)]
    rows.append(('Wearable washout pass', len(cur)))

    cur = cur[cur[f'covs_eligible_fitbit_{task}'].fillna(0) == 1]
    rows.append((f'covs_eligible_fitbit', len(cur)))

    wf_t = wf[wf['task'] == task][['person_id', 'wearable_eligible']].copy()
    cur  = cur.merge(wf_t, on='person_id', how='left')
    cur  = cur[cur['wearable_eligible'] == True]
    rows.append(('Wearable eligible → Analysis Fitbit', len(cur)))
    return rows


def build_bd_cascade(task, cohort, wf):
    rows = []
    base = cohort[cohort['group'] == 'BD'].copy()

    step0_label = f'3c. BD qualifying — {task}'
    attr_row = attrition[
        (attrition['step'] == step0_label) & (attrition['group'] == 'BD')
    ]
    n0 = int(attr_row['n_after'].values[0]) if len(attr_row) > 0 else \
         int((base[f'bd_eligible_{task}'].fillna(0) == 1).sum())
    rows.append(('BD-eligible', n0))

    cur = base[base[f'bd_eligible_{task}'].fillna(0) == 1].copy()

    cur = cur[cur[f'etm_washout_pass_{task}'].fillna(False).astype(bool)]
    rows.append(('EtM washout pass', len(cur)))

    qc  = qc_pass_mask(cur, task)
    cur = cur[qc]
    rows.append(('QC pass', len(cur)))

    cur = cur[cur[f'covs_eligible_normative_{task}']]
    rows.append(('covs_eligible_normative_{task} → Analysis Normative', len(cur)))

    cur = cur[cur[f'fitbit_sufficient_{task}'].fillna(False).astype(bool)]
    rows.append(('Fitbit sufficient', len(cur)))

    cur = cur[cur[f'wearable_washout_pass_{task}'].fillna(False).astype(bool)]
    rows.append(('Wearable washout pass', len(cur)))

    cur = cur[cur[f'covs_eligible_fitbit_{task}'].fillna(0) == 1]
    rows.append((f'covs_eligible_fitbit', len(cur)))

    wf_t = wf[wf['task'] == task][['person_id', 'wearable_eligible']].copy()
    cur  = cur.merge(wf_t, on='person_id', how='left')
    cur  = cur[cur['wearable_eligible'] == True]
    rows.append(('Wearable eligible → Analysis Fitbit', len(cur)))
    return rows


def build_hc_cascade(task, cohort, wf):
    rows = []
    base = cohort[cohort['group'] == 'HC'].copy()

    attr_hc = attrition[
        (attrition['group'] == 'HC') &
        attrition['step'].str.startswith('3b.')
    ]
    n0 = int(attr_hc['n_after'].values[0]) if len(attr_hc) > 0 else len(base)
    rows.append(('HC completers (any task)', n0))

    qc  = qc_pass_mask(base, task)
    cur = base[qc].copy()
    rows.append(('QC pass', len(cur)))

    cur = cur[cur[f'covs_eligible_normative_{task}']]
    rows.append(('covs_eligible_normative_{task} → HC normative model', len(cur)))

    cur = cur[cur[f'fitbit_sufficient_{task}'].fillna(False).astype(bool)]
    rows.append(('Fitbit sufficient', len(cur)))

    cur = cur[cur[f'covs_eligible_fitbit_{task}'].fillna(0) == 1]
    rows.append(('covs_eligible_fitbit', len(cur)))

    wf_t = wf[(wf['task'] == task) & (wf['group'] == 'HC')][
        ['person_id', 'wearable_eligible']
    ].copy()
    cur  = cur.merge(wf_t, on='person_id', how='left')
    cur  = cur[cur['wearable_eligible'] == True]
    rows.append(('Wearable eligible → HC Analysis Fitbit (sensitivity arm)', len(cur)))
    return rows


step_labels_mdd = [r[0] for r in build_mdd_cascade(ETM_TASKS[0], cohort, wf)]
step_labels_bd  = [r[0] for r in build_bd_cascade(ETM_TASKS[0], cohort, wf)]
step_labels_hc  = [r[0] for r in build_hc_cascade(ETM_TASKS[0], cohort, wf)]

mdd_data = {task: [_safe_n(r[1]) for r in build_mdd_cascade(task, cohort, wf)]
            for task in ETM_TASKS}
bd_data  = {task: [_safe_n(r[1]) for r in build_bd_cascade(task, cohort, wf)]
            for task in ETM_TASKS}
hc_data  = {task: [_safe_n(r[1]) for r in build_hc_cascade(task, cohort, wf)]
            for task in ETM_TASKS}

attrition_mdd = pd.DataFrame(mdd_data, index=step_labels_mdd)
attrition_bd  = pd.DataFrame(bd_data,  index=step_labels_bd)
attrition_hc  = pd.DataFrame(hc_data,  index=step_labels_hc)

print('=== MDD Attrition Cascade ===')
print(attrition_mdd.to_string())
print('\n=== BD Attrition Cascade ===')
print(attrition_bd.to_string())
print('\n=== HC Attrition Cascade ===')
print(attrition_hc.to_string())

save_res(RES_OUT, 'attrition_mdd.csv', OVERWRITE, attrition_mdd.reset_index(), file_type='csv')
save_res(RES_OUT, 'attrition_bd.csv',  OVERWRITE, attrition_bd.reset_index(),  file_type='csv')
save_res(RES_OUT, 'attrition_hc.csv',  OVERWRITE, attrition_hc.reset_index(),  file_type='csv')

## Task 2: Demographic Tables (Table 1, Table 3, eTable 5)

Table 1: MDD and BD normative-deviation samples. Table 3: MDD wearable sample. eTable 5: NCC
normative sample. The BD and NCC wearable samples are also tabulated (descriptive only).

In [ ]:
# 2a: HC normative-model sample (eTable 5)
race_cats = sorted(set().union(*(
    set(models[t][2]['race'].dropna().unique()) for t in ETM_TASKS
)))
eth_cats = sorted(set().union(*(
    set(models[t][2]['ethnicity'].dropna().unique()) for t in ETM_TASKS
)))

hc_rows = []
for task in ETM_TASKS:
    cfg = TASK_CONFIG[task]
    hc  = models[task][2]
    hc_rows.append({
        'Task':            cfg['label'],
        'N':               len(hc),
        'Age (years)':     desc_stats(hc[f'age_at_etm_{task}']),
        'Female sex':      pct_female(hc),
        'Education (0-7)': desc_stats(hc['education_ordinal']),
        'Touchscreen device': pct_touch(hc, task),
        **category_rows(hc, 'race', 'Race', race_cats),
        **category_rows(hc, 'ethnicity', 'Ethnicity', eth_cats),
        'Score':           desc_stats(hc[cfg['outcome']]),
    })

demo_hc = pd.DataFrame(hc_rows).set_index('Task').T
print('=== Table: HC Normative Model Sample -- Cognition-only Analyses ===')
print(demo_hc.to_string())
save_res(RES_OUT, 'demo_hc.csv', OVERWRITE, demo_hc.reset_index(), file_type='csv')

In [ ]:
# Demographic table for a normative-deviation sample (MDD or BD)
def build_demo_ab(sample_dict):
    race_cats = sorted(set().union(*(
        set(sample_dict[t]['race'].dropna().unique()) for t in ETM_TASKS
    )))
    eth_cats = sorted(set().union(*(
        set(sample_dict[t]['ethnicity'].dropna().unique()) for t in ETM_TASKS
    )))

    rows = []
    for task in ETM_TASKS:
        cfg     = TASK_CONFIG[task]
        df      = sample_dict[task]
        cci_col = f'cci_{task}'
        ep_col  = f'n_prior_episodes_{task}'
        row = {
            'Task':            cfg['label'],
            'N':               len(df),
            'Age (years)':     desc_stats(df[f'age_at_etm_{task}']),
            'Female sex':      pct_female(df),
            'Education (0-7)': desc_stats(df['education_ordinal']),
            'Touchscreen device': pct_touch(df, task),
            'BMI (kg/m2)':     bmi_desc(df, task),
            'CCI':             desc_stats(df[cci_col].dropna()) if cci_col in df.columns else 'N/A',
            'Prior episodes':  desc_stats(df[ep_col].dropna()) if ep_col in df.columns else 'N/A',
        }
        row.update(category_rows(df, 'race', 'Race', race_cats))
        row.update(category_rows(df, 'ethnicity', 'Ethnicity', eth_cats))
        row.update(season_rows(df, task))
        row['Score'] = desc_stats(df[cfg['outcome']])
        for cls in MED_CLASSES:
            row[f'Med: {MED_LABELS[cls]}'] = med_pct(df, cls, task)
        rows.append(row)
    return pd.DataFrame(rows).set_index('Task').T


# 2b: MDD normative-deviation sample (Table 1)
demo_mdd_ab = build_demo_ab(ab_mdd)
print('=== Table: MDD Sample -- Cognition-only Analyses ===')
print(demo_mdd_ab.to_string())
save_res(RES_OUT, 'demo_mdd_ab.csv', OVERWRITE, demo_mdd_ab.reset_index(), file_type='csv')

In [ ]:
# 2c: BD normative-deviation sample (Table 1)
demo_bd_ab = build_demo_ab(ab_bd)
print('=== Table: BD Sample -- Cognition-only Analyses ===')
print(demo_bd_ab.to_string())
save_res(RES_OUT, 'demo_bd_ab.csv', OVERWRITE, demo_bd_ab.reset_index(), file_type='csv')

In [ ]:
# As build_demo_ab, plus the wearable phenotypes, for a wearable-analysis sample
def build_demo_c(sample_dict):
    race_cats = sorted(set().union(*(
        set(sample_dict[t]['race'].dropna().unique()) for t in ETM_TASKS
    )))
    eth_cats = sorted(set().union(*(
        set(sample_dict[t]['ethnicity'].dropna().unique()) for t in ETM_TASKS
    )))

    rows = []
    for task in ETM_TASKS:
        cfg     = TASK_CONFIG[task]
        df      = sample_dict[task]
        cci_col = f'cci_{task}'
        ep_col  = f'n_prior_episodes_{task}'
        row = {
            'Task':            cfg['label'],
            'N':               len(df),
            'Age (years)':     desc_stats(df[f'age_at_etm_{task}']),
            'Female sex':      pct_female(df),
            'Education (0-7)': desc_stats(df['education_ordinal']),
            'Touchscreen device': pct_touch(df, task),
            'BMI (kg/m2)':     bmi_desc(df, task),
            'CCI':             desc_stats(df[cci_col].dropna()) if cci_col in df.columns else 'N/A',
            'Prior episodes':  desc_stats(df[ep_col].dropna()) if ep_col in df.columns else 'N/A',
        }
        row.update(category_rows(df, 'race', 'Race', race_cats))
        row.update(category_rows(df, 'ethnicity', 'Ethnicity', eth_cats))
        row.update(season_rows(df, task))
        row['Score'] = desc_stats(df[cfg['outcome']])
        for cls in MED_CLASSES:
            row[f'Med: {MED_LABELS[cls]}'] = med_pct(df, cls, task)
        for pheno in PHENOTYPES:
            if pheno in df.columns:
                n_avail = df[pheno].notna().sum()
                row[PHENO_LABELS[pheno]] = (
                    desc_stats(df[pheno].dropna()) + f' [N={n_avail}]'
                ) if n_avail >= 20 else SUPPRESSED
            else:
                row[PHENO_LABELS[pheno]] = 'N/A'
        rows.append(row)
    return pd.DataFrame(rows).set_index('Task').T


# 2d: MDD wearable sample (Table 3)
demo_mdd_c = build_demo_c(c_mdd)
print('=== Table: MDD Sample -- Cognition-Wearable Analyses ===')
print(demo_mdd_c.to_string())
save_res(RES_OUT, 'demo_mdd_c.csv', OVERWRITE, demo_mdd_c.reset_index(), file_type='csv')

In [ ]:
# 2e: BD wearable sample (not modelled; descriptive only)
demo_bd_c = build_demo_c(c_bd)
print('=== Table: BD Sample -- Cognition-Wearable Analyses ===')
print(demo_bd_c.to_string())
save_res(RES_OUT, 'demo_bd_c.csv', OVERWRITE, demo_bd_c.reset_index(), file_type='csv')

In [ ]:
# 2f: HC wearable sample (the HC sensitivity arm in 04b)
demo_hc_c = build_demo_c(c_hc)
print('=== Table: HC Sample -- Cognition-Wearable Analyses ===')
print(demo_hc_c.to_string())
save_res(RES_OUT, 'demo_hc_c.csv', OVERWRITE, demo_hc_c.reset_index(), file_type='csv')

## Task 3: Normative Model Diagnostics (eFigure 2)

Standardised residuals of the HC normative regression for each task, against N(0, 1).

In [ ]:
# 3a: Standardised residuals of the HC normative model
x_grid = np.linspace(-4, 4, 200)
fig, axes = plt.subplots(1, 4, figsize=(16, 4), dpi=DPI)

task_display_labels = {
    'gradcpt': 'GradCPT',
    'flanker': 'Flanker',
    'delaydiscounting': 'DD',
    'emorecog': 'EmoRecog'
}

for ax, task in zip(axes, ETM_TASKS):
    m_norm, _, hc = models[task]
    std_resid = m_norm.resid / np.sqrt(m_norm.mse_resid)
    n  = len(std_resid)
    r2 = m_norm.rsquared

    ax.hist(std_resid, bins=30, density=True, alpha=0.4, color=GROUP_COLORS['HC'], label='Residuals')
    sns.kdeplot(std_resid, ax=ax, color=GROUP_COLORS['HC'], linewidth=1.5)
    ax.plot(x_grid, stats.norm.pdf(x_grid), 'k--', linewidth=1.5, label='N(0,1)')

    display_label = task_display_labels.get(task, task)
    ax.set_title(f"{display_label}\nN={n:,}  R²={r2:.3f}")
    
    ax.set_xlabel('Standardised residual')
    ax.set_ylabel('Density')
    ax.legend(fontsize=7)

plt.tight_layout()
save_res(RES_OUT, 'hc_residuals.png', OVERWRITE, fig, dpi=DPI)
plt.show()

In [ ]:
# 3b: Normality tests on standardised residuals
for task in ETM_TASKS:
    m_norm = models[task][0]
    resid  = m_norm.resid / np.sqrt(m_norm.mse_resid)
    n      = len(resid)
    if n <= 5000:
        stat, p   = stats.shapiro(resid)
        test_name = 'Shapiro-Wilk'
    else:
        stat, p   = stats.kstest(resid, 'norm')
        test_name = 'K-S vs N(0,1)'
    print(f'{task}: {test_name}  W/D={stat:.4f}  p={p:.4g}  N={n:,}')

## Task 4: D Score Distributions

Distribution of `D` in MDD and BD per task (descriptive).

In [ ]:
a_res = model_results[model_results['analysis'] == 'Normative'].copy()

fig, axes = plt.subplots(1, 4, figsize=(16, 5), sharey=False, dpi=300)

task_display_labels = {
    'gradcpt': 'GradCPT',
    'flanker': 'Flanker',
    'delaydiscounting': 'DD',
    'emorecog': 'EmoRecog'
}

# Direction of better performance on each task's D scale
task_suffixes = {
    'gradcpt': '(higher better)',
    'emorecog': '(higher better)',
    'flanker': '(lower better)',
    'delaydiscounting': '(lower better)'
}

for col_idx, task in enumerate(ETM_TASKS):
    ax = axes[col_idx]
    display_label = task_display_labels.get(task, task)
    suffix = task_suffixes.get(task, '')
    
    title_parts = []
    
    for grp, sample_dict in [('MDD', ab_mdd), ('BD', ab_bd)]:
        df = sample_dict[task]
        n = len(df)

        if n < 20:
            title_parts.append(f"{grp}: N<20 (suppressed)")
            continue

        mean_d = df['D'].mean()
        title_parts.append(f"{grp}: N={n}")

        # common_norm=False ensures each group's KDE area integrates to 1 independently
        sns.kdeplot(
            df['D'], 
            ax=ax, 
            color=GROUP_COLORS[grp], 
            alpha=1, 
            common_norm=False,
            label=f"{grp} (mean D={mean_d:+.2f})",
        )

    ax.set_title(f"{display_label}\n" + " | ".join(title_parts), fontsize=10)    
    ax.set_xlabel(f'$D$ score {suffix}')
    ax.set_ylabel('Density')
    ax.legend(fontsize=12, loc='upper right')

plt.tight_layout()
save_res(RES_OUT, 'd_score_distributions.png', OVERWRITE, fig, dpi=DPI)
plt.show()

## Task 5: Wearable Feature Distributions (eFigure 1)

The four phenotypes in the MDD wearable sample and the NCC comparison sample. Each
participant's value is averaged over the tasks for which they were analysed.

In [ ]:
PHENO_PLOT_TITLES = {
    'mean_steps':          'Steps',
    'mean_waso':           'WASO (min)',
    'mean_sleep_duration': 'TST (min)',
    'STV':                 'STV (h)',
}

fig, axes = plt.subplots(1, len(PHENOTYPES), figsize=(5 * len(PHENOTYPES), 4), dpi=300)

for ax, pheno in zip(axes, ['mean_steps', 'mean_waso', 'mean_sleep_duration', 'STV', ]):
    for grp, color in GROUP_COLORS.items():
        # BD is not in the wearable analysis
        if grp == 'BD':
            continue
            
        sub = wf_avg.loc[wf_avg['group'] == grp, pheno].dropna()
        if len(sub) < 20:
            continue
            
        sns.kdeplot(sub, 
                    ax=ax, 
                    label=f"{'NCC' if grp == 'HC' else grp} (N={len(sub):,})",  # NCC in the manuscript
                    color=color, 
                    alpha=1
                   )
        
    ax.set_xlabel(PHENO_LABELS[pheno])
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
    ax.set_title(PHENO_PLOT_TITLES[pheno])
plt.tight_layout()
save_res(RES_OUT, 'wearable_feature_distributions.png', OVERWRITE, fig, DPI)
plt.show()


## Task 6: Results Tables (Table 2, Table 4, eTable 6) and Figure 1

Normative deviation (Table 2, Figure 1), phenotype associations in MDD (Table 4) and in the NCC
comparison arm (eTable 6, nominal p).

In [ ]:
# Table A: normative deviation (Table 2)
a_res = model_results[model_results['analysis'] == 'Normative'].copy()

group_order = ['MDD', 'BD', 'contrast']

table_a_rows = []
for task in ETM_TASKS:
    for grp in group_order:
        row = a_res[(a_res['task'] == task) & (a_res['group'] == grp)]
        if len(row) == 0:
            continue
        row = row.iloc[0]
        suppressed = bool(row.get('suppressed', False))
        if grp == 'MDD':
            n_label = str(row['n_mdd'])
        elif grp == 'BD':
            n_label = str(row['n_bd'])
        else:
            n_label = f"MDD={row['n_mdd']}, BD={row['n_bd']}"
        table_a_rows.append({
            'Task':        TASK_CONFIG[task]['label'],
            'Group':       grp,
            'N':           n_label,
            'beta':        row['beta']       if not suppressed else 'suppressed',
            'ci_lo':       row['ci_lo']      if not suppressed else 'suppressed',
            'ci_hi':       row['ci_hi']      if not suppressed else 'suppressed',
            'effect_size': row['effect_size'] if not suppressed else 'suppressed',
            'p_value':     row['p_value'],
            'p_bh':        row['p_bh'],
            'significant': row['significant'],
        })

table_a = pd.DataFrame(table_a_rows)
with pd.option_context('display.float_format', '{:.3f}'.format, 'display.max_rows', 50):
    print('=== Table 2 — Normative Deviation: cognition-only analyses ===')
    print(table_a.to_string(index=False))

In [ ]:
# Forest plot: normative deviations (Figure 1)
D_WORSE_DIRECTION = {
    'gradcpt':          'lower worse',
    'flanker':          'higher worse',
    'delaydiscounting': 'higher worse',
    'emorecog':         'lower worse',
}
CONTRAST_LABEL = 'BD − MDD'
CONTRAST_COLOR = '#7f7f7f'
MARKER_SIZE    = {'MDD': 110, 'BD': 110, 'contrast': 160}

forest_rows = []
x = 0  # marker position along the x-axis
for task in ETM_TASKS:
    for grp in group_order:
        row = a_res[(a_res['task'] == task) & (a_res['group'] == grp)]
        if len(row) == 0:
            continue
        row = row.iloc[0]
        suppressed = bool(row.get('suppressed', False))
        if grp == 'MDD':
            n = row['n_mdd']
        elif grp == 'BD':
            n = row['n_bd']
        else:
            n = None
        forest_rows.append({
            'task': task, 'grp': grp, 'x': x,
            'beta': row['beta'], 'ci_lo': row['ci_lo'], 'ci_hi': row['ci_hi'],
            'n': n, 'suppressed': suppressed,
        })
        x += 1
    x += 0.8  # Gap between task blocks along x-axis

forest_df = pd.DataFrame(forest_rows)

fig, ax = plt.subplots(figsize=(8.5, 7))

# Vertical zebra striping + task labels placed below the X-axis
for i, task in enumerate(ETM_TASKS):
    block = forest_df[forest_df['task'] == task]
    x_left, x_right = block['x'].min() - 0.5, block['x'].max() + 0.5
    if i % 2 == 1:
        ax.axvspan(x_left, x_right, color='grey', alpha=0.06, zorder=0)
    x_mid = block['x'].mean()
    ax.annotate(
        f"{TASK_CONFIG[task]['label_short']}\n({D_WORSE_DIRECTION[task]})",
        xy=(x_mid, -0.02), xycoords=ax.get_xaxis_transform(),
        ha='center', va='top', fontsize=9, fontweight='bold', linespacing=1.3,
    )

# Horizontal reference line at y = 0
ax.axhline(0, color='black', linewidth=1, linestyle='--', zorder=1)

for _, r in forest_df.iterrows():
    if r['suppressed']:
        continue
    color  = GROUP_COLORS.get(r['grp'], CONTRAST_COLOR)
    marker = 'D' if r['grp'] == 'contrast' else 'o'
    
    # Plot vertically: x is the group position, y is the beta value
    ax.errorbar(
        r['x'], r['beta'],
        yerr=[[r['beta'] - r['ci_lo']], [r['ci_hi'] - r['beta']]],
        fmt='none', ecolor=color, elinewidth=1.3, capsize=3, zorder=2,
    )
    ax.scatter(
        r['x'], r['beta'], s=MARKER_SIZE[r['grp']], marker=marker,
        facecolor=color, edgecolor=color, linewidth=1, zorder=3,
    )
    # Annotate N values above the markers
    if r['n'] is not None and pd.notna(r['n']):
        ax.annotate(
            f"N={r['n']:,.0f}", xy=(r['x'], r['ci_hi']),
            xytext=(0, 4), textcoords='offset points',
            ha='center', va='bottom', fontsize=11, color='dimgrey', rotation=45
        )

# Axis configuration
ax.set_xticks([])
ax.set_xlim(forest_df['x'].min() - 0.8, forest_df['x'].max() + 0.8)

custom_ticks = [-0.2, -0.1, 0.0, 0.1, 0.2]
ax.set_yticks(custom_ticks)
ax.set_ylabel(r"Normative $D$ Score")

legend_handles = [
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=GROUP_COLORS['MDD'],
               markeredgecolor=GROUP_COLORS['MDD'], markersize=8, label='MDD'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=GROUP_COLORS['BD'],
               markeredgecolor=GROUP_COLORS['BD'], markersize=8, label='BD'),
    plt.Line2D([0], [0], marker='D', color='w', markerfacecolor=CONTRAST_COLOR,
               markeredgecolor=CONTRAST_COLOR, markersize=7, label=CONTRAST_LABEL),
]
ax.legend(handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, -0.12),
          ncol=3, frameon=False, fontsize=9)

fig.subplots_adjust(left=0.15, right=0.95, top=0.92, bottom=0.22)
save_res(RES_OUT, 'forest_plot_normative.png', OVERWRITE, fig, dpi=DPI)
plt.show()

In [ ]:
# Table B: rest-activity phenotypes in MDD, main pool (Table 4)
b_res = model_results[model_results['analysis'] == 'Fitbit'].copy()

table_b_rows = []
for grp in ['MDD']:
    for task in ETM_TASKS:
        for pheno in PHENOTYPES:
            row = b_res[
                (b_res['task'] == task) &
                (b_res['phenotype'] == pheno) &
                (b_res['group'] == grp)
            ]
            if len(row) == 0:
                continue
            row = row.iloc[0]
            suppressed = bool(row.get('suppressed', False))
            table_b_rows.append({
                'Group':       grp,
                'Task':        TASK_CONFIG[task]['label'],
                'Phenotype':   PHENO_LABELS[pheno],
                'n_mdd':       row['n_mdd'],
                'n_hc':        row['n_hc'],
                'beta':        row['beta']       if not suppressed else 'suppressed',
                'se_hc3':      row['se_hc3']     if not suppressed else 'suppressed',
                'ci_lo':       row['ci_lo']      if not suppressed else 'suppressed',
                'ci_hi':       row['ci_hi']      if not suppressed else 'suppressed',
                'p_value':     row['p_value'],
                'p_bh':        row['p_bh'],
                'significant': row['significant'],
            })

table_b = pd.DataFrame(table_b_rows)
with pd.option_context('display.float_format', '{:.3f}'.format, 'display.max_rows', 100):
    print('=== Table 4 — Cognition-Wearable Analyses (MDD only, main pool) ===')
    print(table_b.to_string(index=False))

In [ ]:
# Table B, NCC comparison arm (eTable 6): nominal p, effect size Pearson r
table_b_hc_rows = []
for task in ETM_TASKS:
    for pheno in PHENOTYPES:
        row = hc_model_results[
            (hc_model_results['task'] == task) & (hc_model_results['phenotype'] == pheno)
        ]
        if len(row) == 0:
            continue
        row = row.iloc[0]
        suppressed = bool(row.get('suppressed', False))
        table_b_hc_rows.append({
            'Task':                TASK_CONFIG[task]['label'],
            'Phenotype':           PHENO_LABELS[pheno],
            'n_hc':                row['n_hc'],
            'beta':                row['beta']        if not suppressed else 'suppressed',
            'se_hc3':              row['se_hc3']      if not suppressed else 'suppressed',
            'ci_lo':               row['ci_lo']       if not suppressed else 'suppressed',
            'ci_hi':               row['ci_hi']       if not suppressed else 'suppressed',
            'effect_size':         row['effect_size'] if not suppressed else 'suppressed',
            'p_value':             row['p_value'],
            'significant':         row['significant'],
            'mdes_r_nominal':      row['mdes_r_nominal'],
            'mdes_r2_nominal':     row['mdes_r2_nominal'],
        })

table_b_hc_sensitivity = pd.DataFrame(table_b_hc_rows)
with pd.option_context('display.float_format', '{:.3f}'.format, 'display.max_rows', 100):
    print('=== eTable 6 — Cognition-Wearable Analyses in HC sample (nominal p, uncorrected) ===')
    print(table_b_hc_sensitivity.to_string(index=False))


In [ ]:
# Save results tables to GCS
save_parquet(table_a, 'results/table_a.parquet')
save_parquet(table_b, 'results/table_b.parquet')
save_parquet(table_b_hc_sensitivity, 'results/table_b_hc_sensitivity.parquet')
print('Tables A, B, and B (HC sensitivity) saved to GCS.')

## Task 7: BD-Only Power and U-Shaped TST (eTable 8)

1. The MDES a BD-only phenotype analysis would have had (from `04c`).
2. The U-shaped TST check (from `04b`), nominal p.

In [ ]:
mdes_stratified_posthoc = load_parquet('results/mdes_stratified_posthoc.parquet')
sleep_duration_ushape    = load_parquet('results/sleep_duration_ushape_sensitivity.parquet')


In [ ]:
# MDES of a BD-only phenotype analysis (from 04c)
print('=== Post-hoc power: BD-only rest-activity MDES (D ~ P_z), nominal alpha = 0.05 ===')
print(mdes_stratified_posthoc.set_index(['phenotype', 'task']).to_string())


### U-Shaped TST: HC and MDD (eTable 8)

`D ~ P_z + I(P_z ** 2)` for TST. The figure shows mean `D` in deciles of `P_z` (bins of at least
20 participants) with the fitted quadratic.

In [ ]:
print('=== Sleep-duration quadratic (U-shape) sensitivity — HC & MDD, nominal p only ===')
with pd.option_context('display.float_format', '{:.4f}'.format, 'display.max_rows', 20):
    print(
        sleep_duration_ushape[
            ['task', 'sample', 'n', 'suppressed', 'r_lin', 'p_lin',
             'b2', 'se2', 'p2', 'vertex', 'vertex_in_range', 'vertex_pctl',
             'tails_worse_than_mid']
        ]
        .sort_values(['sample', 'task'])
        .to_string(index=False)
    )


In [ ]:
# Binned D vs. residualised sleep-duration (P_z), with a locally-refit quadratic overlay.
# Only bins with n >= 20 are plotted (AoU governance — no participant-level scatter).
_PHENOTYPE = 'mean_sleep_duration'

def _residualize_and_fit(df, task):
    df        = df.copy()
    df['P_z'] = residualise_phenotype(df, task, _PHENOTYPE)
    m_quad    = smf.ols('D ~ P_z + I(P_z ** 2)', data=df).fit(cov_type='HC3')
    return df, m_quad

fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharex=False)

for col_idx, task in enumerate(ETM_TASKS):
    cfg = TASK_CONFIG[task]

    # Both samples carry D: out-of-sample for MDD, the in-sample residual for HC,
    # exactly as in 04b's U-shape check.
    df_mdd = phenotype_complete(c_mdd[task], _PHENOTYPE)
    df_hc  = phenotype_complete(c_hc[task],  _PHENOTYPE)

    for row_idx, (sample_label, df) in enumerate([('MDD', df_mdd), ('HC', df_hc)]):
        ax = axes[row_idx, col_idx]
        if len(df) < MIN_CELL:
            ax.text(0.5, 0.5, 'N<20\n(suppressed)', transform=ax.transAxes,
                    ha='center', va='center', fontsize=10)
            ax.set_title(f"{cfg['label']} \u2014 {sample_label}")
            continue

        df_fit, m_quad = _residualize_and_fit(df, task)
        edges = np.unique(np.quantile(df_fit['P_z'], np.linspace(0, 1, 11)))
        binned = (
            df_fit.assign(bin=pd.cut(df_fit['P_z'], edges, include_lowest=True))
                  .groupby('bin', observed=True)
                  .agg(n=('D', 'size'), pz_mid=('P_z', 'mean'), d_mean=('D', 'mean'))
                  .reset_index()
        )
        binned = binned[binned['n'] >= MIN_CELL]

        if not binned.empty:
            ax.scatter(binned['pz_mid'], binned['d_mean'], s=30, zorder=3, color=GROUP_COLORS[sample_label])
            x_line = np.linspace(df_fit['P_z'].min(), df_fit['P_z'].max(), 200)
            y_line = m_quad.predict(pd.DataFrame({'P_z': x_line}))
            ax.plot(x_line, y_line, color=GROUP_COLORS[sample_label], alpha=0.7)

        u_row = sleep_duration_ushape[
            (sleep_duration_ushape['task'] == task) & (sleep_duration_ushape['sample'] == sample_label)
        ]
        if len(u_row) and bool(u_row['vertex_in_range'].values[0]):
            ax.axvline(u_row['vertex'].values[0], color='gray', linestyle='--', alpha=0.6)

        ax.set_title(f"{cfg['label']} \u2014 {sample_label}  (N={len(df):,})")
        ax.set_xlabel('P_z (residualised sleep duration)')
        ax.set_ylabel(f"D score ({cfg['direction']})")

fig.suptitle('Sleep duration \u2014 binned D vs. residualised P_z, quadratic overlay (nominal, uncorrected)',
             fontsize=12, y=1.02)
plt.tight_layout()
save_res(RES_OUT, 'sleep_duration_ushape_binned_fit.png', OVERWRITE, fig)
plt.show()


## Task 8: Antipsychotic Exclusion (eTable 9)

The normative-deviation estimates from `04a` beside the antipsychotic-excluded refit from `04b`
(nominal p), as a table (eTable 9) and an overlaid forest plot (filled: full sample; hollow:
excluded). Also tabulates excluded vs retained participants.

In [ ]:
# ── 8a: Side-by-side, main pool vs antipsychotic-excluded ────────────────────
noap_res    = load_parquet('results/antipsychotic_sensitivity.parquet')
ap_balance  = load_parquet('results/antipsychotic_dropped_vs_retained_balance.parquet')
a_res_main  = model_results[model_results['analysis'] == 'Normative'].copy()

print(f'antipsychotic_sensitivity.parquet                : {len(noap_res):,} rows')
print(f'antipsychotic_dropped_vs_retained_balance.parquet: {len(ap_balance):,} rows')

merged = a_res_main.merge(
    noap_res[['task', 'group', 'beta', 'se_hc3', 'ci_lo', 'ci_hi', 'p_value',
              'beta_full', 'se_full', 'p_full', 'delta_beta',
              'n_mdd', 'n_bd', 'n_dropped_mdd', 'n_dropped_bd', 'suppressed']]
        .rename(columns={
            'beta': 'beta_noap', 'se_hc3': 'se_noap',
            'ci_lo': 'ci_lo_noap', 'ci_hi': 'ci_hi_noap', 'p_value': 'p_noap',
            'n_mdd': 'n_mdd_noap', 'n_bd': 'n_bd_noap',
            'suppressed': 'suppressed_noap',
        }),
    on=['task', 'group'], how='left', validate='one_to_one',
)
assert len(merged) == len(a_res_main) == 12, f'merge changed row count: {len(merged)}'

# 04b refits the unrestricted model itself; it must reproduce 04a, or the two notebooks
# built different samples.
_cmp = merged[~merged['suppressed'].fillna(False) & ~merged['suppressed_noap'].fillna(False)]
assert np.allclose(_cmp['beta'], _cmp['beta_full'], atol=1e-9), (
    "04b's beta_full does not reproduce 04a's beta — the Normative sample reconstruction in "
    "04b_modelling_sensitivity.ipynb has diverged from 04a_modelling_main.ipynb."
)
print(f'\nCross-check OK: 04b\'s unrestricted refit reproduces 04a on all {len(_cmp)} estimands '
      f'(max |Δ| = {np.abs(_cmp["beta"] - _cmp["beta_full"]).max():.2e}).')

rows = []
for task in ETM_TASKS:
    for grp in group_order:
        r = merged[(merged['task'] == task) & (merged['group'] == grp)]
        if len(r) == 0:
            continue
        r = r.iloc[0]
        if grp == 'MDD':
            n_full, n_keep, n_drop = r['n_mdd'], r['n_mdd_noap'], r['n_dropped_mdd']
            n_full_s, n_keep_s, n_drop_s = f'{n_full:,.0f}', f'{n_keep:,.0f}', f'{n_drop:,.0f}'
        elif grp == 'BD':
            n_full, n_keep, n_drop = r['n_bd'], r['n_bd_noap'], r['n_dropped_bd']
            n_full_s, n_keep_s, n_drop_s = f'{n_full:,.0f}', f'{n_keep:,.0f}', f'{n_drop:,.0f}'
        else:
            n_full_s = f'{r["n_mdd"]:,.0f}/{r["n_bd"]:,.0f}'
            n_keep_s = f'{r["n_mdd_noap"]:,.0f}/{r["n_bd_noap"]:,.0f}'
            n_drop_s = f'{r["n_dropped_mdd"]:,.0f}/{r["n_dropped_bd"]:,.0f}'
        supp = bool(r['suppressed']) or bool(r['suppressed_noap'])
        rows.append({
            'Task':             TASK_CONFIG[task]['label_short'],
            'Group':            CONTRAST_LABEL if grp == 'contrast' else grp,
            'N (full)':         n_full_s,
            'N dropped':        n_drop_s,
            'N (no AP)':        n_keep_s,
            'Delta (full)':     r['beta']      if not supp else 'suppressed',
            'CI (full)':        f'[{r["ci_lo"]:+.3f}, {r["ci_hi"]:+.3f}]' if not supp else '—',
            'p (full, nom.)':   r['p_full']    if not supp else None,
            'p (full, BH)':     r['p_bh']      if not supp else None,
            'Delta (no AP)':    r['beta_noap'] if not supp else 'suppressed',
            'CI (no AP)':       f'[{r["ci_lo_noap"]:+.3f}, {r["ci_hi_noap"]:+.3f}]' if not supp else '—',
            'p (no AP, nom.)':  r['p_noap']    if not supp else None,
            'Shift':            r['delta_beta'] if not supp else None,
            'Shift / SE_full':  (abs(r['delta_beta']) / r['se_hc3']) if not supp else None,
        })

table_a_ap = pd.DataFrame(rows)
with pd.option_context('display.float_format', '{:.3f}'.format, 'display.max_rows', 50,
                       'display.width', 250):
    print('\n=== eTable 9 — Normative Deviation: full sample vs antipsychotic-excluded ===')
    print(table_a_ap.to_string(index=False))

_max_shift = table_a_ap['Shift'].abs().max()
_max_rel   = table_a_ap['Shift / SE_full'].max()
print(f'\nLargest |shift|: {_max_shift:.3f} Glass\'s Δ units ({_max_rel:.2f} × SE of the main '
      f'estimate).')
print('p (no AP, nom.) compares with p (full, nom.); p (full, BH) is corrected over the 28 tests.')

save_res(RES_OUT, 'table_a_antipsychotic_sensitivity.csv', OVERWRITE, table_a_ap, file_type='csv')
save_parquet(table_a_ap, 'results/table_a_antipsychotic_sensitivity.parquet')

In [ ]:
# ── 8b: Dropped vs retained balance ─────────────────────────────────────────
# Binary rows are proportions of `positive_category`: % female for sex, % True for touch.
# Assert it, so the table cannot silently become % male.
assert 'positive_category' in ap_balance.columns, (
    'antipsychotic_dropped_vs_retained_balance.parquet has no positive_category column. '
    'Re-run 04b_modelling_sensitivity.ipynb before this notebook.'
)
_sex_cats = set(ap_balance.loc[ap_balance['variable'] == 'sex', 'positive_category'].dropna())
assert _sex_cats <= {'Female'}, (
    f'04b reported sex as {_sex_cats} — this table and Table 1 report % female. '
    'Check BINARY_POS_CATEGORY in 04b_modelling_sensitivity.ipynb.'
)

VAR_LABELS = {
    'D':                 'D score',
    'age_at_etm':        'Age (years)',
    'education_ordinal': 'Education (0-7)',
    'bmi':               'BMI (kg/m2)',
    'cci':               'CCI',
    'n_prior_episodes':  'Prior episodes',
    'sex':               'Female sex',
    'touch':             'Touchscreen device',
}
VAR_ORDER = ['D', 'age_at_etm', 'education_ordinal', 'bmi', 'cci',
             'n_prior_episodes', 'sex', 'touch']


def _fmt_center(row, side):
    """side ∈ {'dropped', 'retained'} -> display string; binary rows are already the
    proportion in `positive_category` ('Female' for sex, True for touch)."""
    c, s = row[f'{side}_center'], row[f'{side}_spread']
    if pd.isna(c):
        return '—'
    if row['var_type'] == 'binary':
        return f'{100 * c:.1f}%'
    return f'{c:.2f} ± {s:.2f}'


bal_rows = []
for grp in ['MDD', 'BD']:
    for task in ETM_TASKS:
        for var in VAR_ORDER:
            r = ap_balance[(ap_balance['group'] == grp)
                           & (ap_balance['task'] == task)
                           & (ap_balance['variable'] == var)]
            if len(r) == 0:
                continue
            r = r.iloc[0]
            supp = bool(r['suppressed'])
            bal_rows.append({
                'Group':      grp,
                'Task':       TASK_CONFIG[task]['label_short'],
                'Variable':   VAR_LABELS[var],
                'N dropped':  int(r['n_dropped']),
                'N retained': int(r['n_retained']),
                'Dropped':    _fmt_center(r, 'dropped')  if not supp else 'suppressed',
                'Retained':   _fmt_center(r, 'retained') if not supp else 'suppressed',
                'Test':       r['test'] if not supp else '—',
                'Effect size': r['effect_size'] if not supp else None,
                'p':          r['p_value'] if not supp else None,
                'sig':        '*' if (not supp and bool(r['significant'])) else '',
            })

table_ap_balance = pd.DataFrame(bal_rows)
with pd.option_context('display.float_format', '{:.3f}'.format, 'display.max_rows', 100,
                       'display.width', 250):
    for grp in ['MDD', 'BD']:
        print(f'\n=== Table — Antipsychotic dropped vs retained, {grp} '
              f'(nominal p, no correction) ===')
        print(table_ap_balance[table_ap_balance['Group'] == grp]
              .drop(columns='Group').to_string(index=False))

n_sig  = int((table_ap_balance['sig'] == '*').sum())
n_test = int(table_ap_balance['p'].notna().sum())
print(f'\nNominally significant imbalances: {n_sig} / {n_test} '
      f'(~{0.05 * n_test:.0f} expected by chance at alpha = 0.05, uncorrected).')

# D rows: were the excluded participants more impaired?
print('\n--- D score: were the dropped participants more impaired? ---')
_d = ap_balance[(ap_balance['variable'] == 'D') & ~ap_balance['suppressed']].copy()
print(f'{"Task":<10}{"Group":<7}{"D dropped":>11}{"D retained":>12}{"Cohen d":>10}'
      f'{"p":>10}   {"direction":<22}')
for _, r in _d.sort_values(['group', 'task']).iterrows():
    direction = ('dropped MORE impaired' if r['dropped_more_impaired']
                 else 'dropped LESS impaired')
    print(f'{TASK_CONFIG[r["task"]]["label_short"]:<10}{r["group"]:<7}'
          f'{r["dropped_center"]:>+11.3f}{r["retained_center"]:>+12.3f}'
          f'{r["effect_size"]:>+10.3f}{r["p_value"]:>10.4g}   {direction:<22}')
print('\n`direction` compares signs only (task direction applied); read it with Cohen d and p.')

save_res(RES_OUT, 'table_antipsychotic_balance.csv', OVERWRITE, table_ap_balance, file_type='csv')
save_parquet(table_ap_balance, 'results/table_antipsychotic_balance.parquet')

In [ ]:
# ── 8c: Overlaid forest plot — full sample vs antipsychotic-excluded ────────
# Figure 1's estimates (filled) with the antipsychotic-excluded refit (hollow) beside them.
AP_OFFSET = 0.16   # horizontal separation between the paired markers

forest_ap_rows = []
x = 0
for task in ETM_TASKS:
    for grp in group_order:
        r = merged[(merged['task'] == task) & (merged['group'] == grp)]
        if len(r) == 0:
            continue
        r = r.iloc[0]
        if grp == 'MDD':
            n_drop = r['n_dropped_mdd']
            n_drop_s = f'−{n_drop:,.0f}'
        elif grp == 'BD':
            n_drop = r['n_dropped_bd']
            n_drop_s = f'−{n_drop:,.0f}'
        else:
            n_drop_s = f'−{r["n_dropped_mdd"]:,.0f}/{r["n_dropped_bd"]:,.0f}'
        forest_ap_rows.append({
            'task': task, 'grp': grp, 'x': x,
            'beta': r['beta'], 'ci_lo': r['ci_lo'], 'ci_hi': r['ci_hi'],
            'beta_noap': r['beta_noap'],
            'ci_lo_noap': r['ci_lo_noap'], 'ci_hi_noap': r['ci_hi_noap'],
            'n_drop_s': n_drop_s,
            'suppressed': bool(r['suppressed']) or bool(r['suppressed_noap']),
        })
        x += 1
    x += 0.8

forest_ap_df = pd.DataFrame(forest_ap_rows)

fig, ax = plt.subplots(figsize=(10, 7))

for i, task in enumerate(ETM_TASKS):
    block = forest_ap_df[forest_ap_df['task'] == task]
    x_left, x_right = block['x'].min() - 0.5, block['x'].max() + 0.5
    if i % 2 == 1:
        ax.axvspan(x_left, x_right, color='grey', alpha=0.06, zorder=0)
    ax.annotate(
        f"{TASK_CONFIG[task]['label_short']}\n({D_WORSE_DIRECTION[task]})",
        xy=(block['x'].mean(), -0.02), xycoords=ax.get_xaxis_transform(),
        ha='center', va='top', fontsize=9, fontweight='bold', linespacing=1.3,
    )

ax.axhline(0, color='black', linewidth=1, linestyle='--', zorder=1)

for _, r in forest_ap_df.iterrows():
    if r['suppressed']:
        continue
    color  = GROUP_COLORS.get(r['grp'], CONTRAST_COLOR)
    marker = 'D' if r['grp'] == 'contrast' else 'o'
    size   = MARKER_SIZE[r['grp']]

    # Full sample — filled.
    ax.errorbar(
        r['x'] - AP_OFFSET, r['beta'],
        yerr=[[r['beta'] - r['ci_lo']], [r['ci_hi'] - r['beta']]],
        fmt='none', ecolor=color, elinewidth=1.3, capsize=3, zorder=2,
    )
    ax.scatter(
        r['x'] - AP_OFFSET, r['beta'], s=size, marker=marker,
        facecolor=color, edgecolor=color, linewidth=1, zorder=3,
    )
    # Antipsychotic-excluded — hollow, same colour.
    ax.errorbar(
        r['x'] + AP_OFFSET, r['beta_noap'],
        yerr=[[r['beta_noap'] - r['ci_lo_noap']], [r['ci_hi_noap'] - r['beta_noap']]],
        fmt='none', ecolor=color, elinewidth=1.3, capsize=3, alpha=0.75, zorder=2,
    )
    ax.scatter(
        r['x'] + AP_OFFSET, r['beta_noap'], s=size, marker=marker,
        facecolor='white', edgecolor=color, linewidth=1.6, zorder=3,
    )
    # How many participants the exclusion removed, above the taller of the two CIs.
    ax.annotate(
        r['n_drop_s'], xy=(r['x'], max(r['ci_hi'], r['ci_hi_noap'])),
        xytext=(0, 5), textcoords='offset points',
        ha='center', va='bottom', fontsize=9, color='dimgrey', rotation=45,
    )

ax.set_xticks([])
ax.set_xlim(forest_ap_df['x'].min() - 0.9, forest_ap_df['x'].max() + 0.9)
ax.set_yticks([-0.3, -0.2, -0.1, 0.0, 0.1, 0.2, 0.3])
ax.set_ylabel(r"Normative $D$ Score")

group_handles = [
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=GROUP_COLORS['MDD'],
               markeredgecolor=GROUP_COLORS['MDD'], markersize=8, label='MDD'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=GROUP_COLORS['BD'],
               markeredgecolor=GROUP_COLORS['BD'], markersize=8, label='BD'),
    plt.Line2D([0], [0], marker='D', color='w', markerfacecolor=CONTRAST_COLOR,
               markeredgecolor=CONTRAST_COLOR, markersize=7, label=CONTRAST_LABEL),
]
fill_handles = [
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='dimgrey',
               markeredgecolor='dimgrey', markersize=8, label='Full sample'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='white',
               markeredgecolor='dimgrey', markeredgewidth=1.6, markersize=8,
               label='Antipsychotic-excluded'),
]
leg1 = ax.legend(handles=group_handles, loc='upper center', bbox_to_anchor=(0.30, -0.12),
                 ncol=3, frameon=False, fontsize=9)
ax.add_artist(leg1)
ax.legend(handles=fill_handles, loc='upper center', bbox_to_anchor=(0.80, -0.12),
          ncol=2, frameon=False, fontsize=9)
ax.annotate(
    'Grey labels: participants removed by the exclusion',
    xy=(0.5, -0.20), xycoords='axes fraction', ha='center', va='top',
    fontsize=8, color='dimgrey',
)

fig.subplots_adjust(left=0.13, right=0.97, top=0.94, bottom=0.24)
save_res(RES_OUT, 'forest_plot_normative_antipsychotic.png', OVERWRITE, fig, dpi=DPI)
plt.show()